# Encontro 3 · Demonstração: o mesmo agente, com LangChain

**Para o professor apresentar.** Mesmo propósito do notebook 03, agora com as peças prontas do LangChain. Rode o 03 antes: no fim deste, as quatro rodadas aparecem lado a lado.

O que **não** muda, de propósito: os casos, a régua, as regras de resposta, o contrato da ferramenta e a `camada.py` do Encontro 2, por onde toda busca passa. Assim, o placar compara só o framework.

Este notebook precisa de pacotes a mais, só na máquina do professor:

```
pip install -r encontro-03/requirements-demo.txt
```

Com `PROVEDOR=fake` ele não roda: as classes do LangChain falam direto com o provedor.

In [1]:
import json, os
import agente_busca as ab
import camada
import rag_aurora as r
from comum import medidor, modelo
from comum.resultado import Resultado

casos = ab.carregar_casos()
print(f"Provedor: {modelo.PROVEDOR} · modelo: {modelo.MODELO} · {len(casos)} casos")

Provedor: anthropic · modelo: claude-haiku-4-5-20251001 · 16 casos


## 1 · O modelo, pelo LangChain

No 03, o provedor fica atrás do `comum/modelo.py`. Aqui, cada provedor tem a sua classe.

In [2]:
if modelo.PROVEDOR == "anthropic":
    from langchain_anthropic import ChatAnthropic
    llm = ChatAnthropic(model=modelo.MODELO, max_tokens=1024)
elif modelo.PROVEDOR == "openai_compat":
    from langchain_openai import ChatOpenAI
    extra = {}
    if "openrouter.ai" in os.getenv("BASE_URL", ""):
        extra = {"provider": {"data_collection": os.getenv("OPENROUTER_DATA_COLLECTION", "deny")}}
    llm = ChatOpenAI(model=modelo.MODELO, base_url=os.environ["BASE_URL"], api_key=os.environ["API_KEY"],
                     max_tokens=1024, extra_body=extra)
else:
    raise RuntimeError("Este notebook precisa de PROVEDOR=anthropic ou openai_compat no .env")
llm

ChatAnthropic(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.3.11', 'langchain-anthropic': '1.7.5'}}, profile={'name': 'Claude Haiku 4.5', 'release_date': '2025-10-15', 'last_updated': '2025-10-15', 'open_weights': False, 'max_input_tokens': 200000, 'max_output_tokens': 64000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': False, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_call_streaming': True}, model='claude-haiku-4-5-20251001', max_tokens=1024, anthropic_api_url='https://api.anthropic.com', anthropic_api_key=SecretStr('**********'), model_kwargs={})

## 2 · A busca, com as classes do LangChain

A mesma configuração do notebook 01, agora montada com peças prontas: o vector store do Chroma, o `BM25Retriever` e o `EnsembleRetriever`, que junta os dois pela mesma fusão por posição (RRF) que escrevemos à mão.

Duas coisas continuam sendo nossas: os prefixos `query:`/`passage:` do modelo E5, que o embrulho abaixo preserva, e a regra de vigência.

In [3]:
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.embeddings import Embeddings
try:
    from langchain_classic.retrievers import EnsembleRetriever
except ImportError:
    from langchain.retrievers import EnsembleRetriever


class E5(Embeddings):
    """O mesmo modelo do notebook 01, com os prefixos que o E5 espera."""
    def __init__(self):
        self.e5 = r.Embeddings()
    def embed_documents(self, textos):
        return self.e5.passagens(textos).tolist()
    def embed_query(self, texto):
        return self.e5.consultas([texto])[0].tolist()


chunks = r.chunks_por_estrutura(r.carregar_corpus())
oficiais = [c for c in chunks if c.metadata["autoridade"] == "oficial"]

vetorial = Chroma.from_documents(chunks, embedding=E5(), collection_name="aurora-langchain").as_retriever(
    search_kwargs={"k": 10, "filter": {"autoridade": "oficial"}})
lexico = BM25Retriever.from_documents(oficiais, preprocess_func=r.tokenizar, k=10)
hibrido = EnsembleRetriever(retrievers=[vetorial, lexico], weights=[0.5, 0.5])


def buscar_langchain(pergunta, espaco=None):
    docs = hibrido.invoke(pergunta)
    if espaco:
        docs = [d for d in docs if d.metadata["espaco"] == espaco]
    docs = [d for d, _ in r.so_a_versao_vigente([(d, 0) for d in docs])][:3]   # regra nossa, não do framework
    return {"resultados": [{"id": d.metadata["id"], "titulo": d.metadata["titulo"],
                            "secao": d.metadata.get("secao", ""), "vigencia": d.metadata["vigencia"],
                            "trecho": d.page_content} for d in docs]}


# Troca a implementação atrás do contrato. O contrato e a camada não mudam.
ab.plugar_na_camada(buscar_langchain)
print(camada.executar(ab.NOME, {"pergunta": "erro 809 na VPN"}, ab.CONTRATOS, [])[:400])

C:\Users\cstefano\AppData\Local\Temp\ipykernel_24252\988319187.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{"resultados": [{"id": "ti-vpn-codigos-de-erro", "titulo": "Códigos de erro da VPN", "secao": "Erro 809", "vigencia": "2026-02-03", "trecho": "Códigos de erro da VPN > Erro 809\nA conexão com a VPN foi recusada porque a rede local bloqueou o tráfego da VPN. Isso é comum em redes de hotel e de aeroporto. Ative a opção \"Modo TCP 443\" no cliente de VPN e reconecte. Se não resolver, use o roteamento


## 3 · A ferramenta e a resposta estruturada

A ferramenta do LangChain recebe o mesmo nome, a mesma descrição e o mesmo schema do contrato, e chama a mesma `camada.executar`. A resposta final vem num schema Pydantic com os campos do `Resultado` dos Encontros 1 e 2.

In [4]:
from typing import Literal
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

atual = {"registro": []}       # o registro da rodada em curso, para o placar contar as buscas


def _buscar(pergunta: str, espaco: str | None = None) -> str:
    args = {"pergunta": pergunta, **({"espaco": espaco} if espaco else {})}
    return camada.executar(ab.NOME, args, ab.CONTRATOS, atual["registro"])


ferramenta = StructuredTool.from_function(func=_buscar, name=ab.NOME, description=ab.CONTRATO["descricao"],
                                          args_schema=ab.CONTRATO["parametros"])


class RespostaAurora(BaseModel):
    """Resposta final ao colaborador."""
    desfecho: Literal["responder", "escalar", "nao_sei"]
    fontes: list[str] = Field(description="ids das páginas que sustentam a resposta")
    resposta: str = Field(description="texto curto para o colaborador")


def registrar_uso(mensagens):
    """Passa os tokens de cada resposta do modelo para o medidor do projeto, como no notebook 03."""
    for m in mensagens:
        uso = getattr(m, "usage_metadata", None)
        if uso:
            medidor.registrar(uso.get("input_tokens", 0), uso.get("output_tokens", 0), 0.0)

## 4 · Arquitetura A, com LangChain: uma chain

Busca, prompt e modelo encadeados. A busca continua passando pela camada, para contar igual ao 03.

In [5]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", ab.REGRAS),
    ("human", "Documentos encontrados pela busca:\n{trechos}\n\nPergunta do colaborador: {pergunta}"),
])
cadeia = prompt | llm.with_structured_output(RespostaAurora, include_raw=True)


def fixo_langchain(pergunta, registro):
    trechos = camada.executar(ab.NOME, {"pergunta": pergunta}, ab.CONTRATOS, registro)
    saida = cadeia.invoke({"trechos": trechos, "pergunta": pergunta})
    registrar_uso([saida["raw"]])
    return Resultado.de_dict(saida["parsed"].model_dump()) if saida["parsed"] else Resultado("invalido")

## 5 · Arquitetura B, com LangChain: `create_agent`

O loop que no 03 ocupou vinte linhas vira uma chamada. O limite de passos vira `recursion_limit`.

In [6]:
from langchain.agents import create_agent

SISTEMA_LC = ab.REGRAS + """

Você tem uma ferramenta de busca nas políticas da Aurora. Decida se precisa dela: busque quando a
resposta depende de uma regra da empresa, e não busque para cumprimentos ou agradecimentos.
Se os trechos não responderem, você pode buscar de novo com outras palavras."""

agente_lc = create_agent(model=llm, tools=[ferramenta], system_prompt=SISTEMA_LC, response_format=RespostaAurora)


def agente_langchain(pergunta, registro):
    atual["registro"] = registro
    saida = agente_lc.invoke({"messages": [{"role": "user", "content": pergunta}]},
                             config={"recursion_limit": 2 * ab.MAX_PASSOS + 1})
    registrar_uso(saida["messages"])
    return Resultado.de_dict(saida["structured_response"].model_dump())

In [7]:
# Vendo o agente decidir: as mensagens que o LangChain guardou.
atual["registro"] = []
saida = agente_lc.invoke({"messages": [{"role": "user", "content": "Estou com o erro 809 na VPN, o que eu faço?"}]})
for m in saida["messages"]:
    chamadas = getattr(m, "tool_calls", None)
    print(f"{type(m).__name__:<14}", chamadas if chamadas else str(m.content)[:120])
saida["structured_response"]

HumanMessage   Estou com o erro 809 na VPN, o que eu faço?
AIMessage      [{'name': 'buscar_politicas', 'args': {'pergunta': 'erro 809 VPN', 'espaco': 'wiki-ti'}, 'id': 'toolu_01Wv4nqeMmFbxi7g2T6AnqwE', 'type': 'tool_call'}]
ToolMessage    {"resultados": [{"id": "ti-vpn-codigos-de-erro", "titulo": "Códigos de erro da VPN", "secao": "Erro 809", "vigencia": "2
AIMessage      {"desfecho": "responder", "fontes": ["ti-vpn-codigos-de-erro"], "resposta": "O erro 809 significa que a rede local bloqu


RespostaAurora(desfecho='responder', fontes=['ti-vpn-codigos-de-erro'], resposta='O erro 809 significa que a rede local bloqueou o tráfego da VPN (comum em redes de hotel e aeroporto). Tente: 1) Ative a opção "Modo TCP 443" no cliente de VPN e reconecte. 2) Se não resolver, use o roteamento pelo celular. Se o erro persistir, abra um chamado no portal de chamados de TI informando o código 809.')

## 6 · As quatro rodadas lado a lado

In [8]:
print("Busca fixa (LangChain)"); fixo_lc = ab.rodar(fixo_langchain, casos, "busca fixa · LangChain")
print("\nAgente que decide (LangChain)"); decide_lc = ab.rodar(agente_langchain, casos, "agente decide · LangChain")

Busca fixa (LangChain)
  N1  ok   buscas=1  ok
  N2  ok   buscas=1  ok
  N4  ok   buscas=1  ok
  N7  ok   buscas=1  ok
  E1  ok   buscas=1  ok
  E2  ok   buscas=1  ok
  C1  ok   buscas=1  ok
  C2  ok   buscas=1  ok
  T1  ok   buscas=1  ok
  T2  ok   buscas=1  ok
  A1  ok   buscas=1  ok
  S1  ok   buscas=1  ok
  S2  ok   buscas=1  ok
  X1  ok   buscas=1  ok
  P1  ok   buscas=1  ok
  P2  ok   buscas=1  ok

Agente que decide (LangChain)
  N1  ok   buscas=1  ok
  N2  ok   buscas=1  ok
  N4  ok   buscas=1  ok
  N7  ok   buscas=1  ok
  E1  ok   buscas=1  ok
  E2  ok   buscas=1  ok
  C1  ok   buscas=1  ok
  C2  ok   buscas=1  ok
  T1  ok   buscas=1  ok
  T2  ok   buscas=1  ok
  A1  ok   buscas=1  ok
  S1  ok   buscas=2  ok
  S2  ok   buscas=1  ok
  X1  ok   buscas=1  ok
  P1  ok   buscas=0  ok
  P2  ok   buscas=0  ok


In [9]:
do_03 = ab.carregar_ultimas()          # as rodadas salvas pelo notebook 03
ab.placar(*do_03, fixo_lc, decide_lc)
print()
ab.detalhe(*do_03, fixo_lc, decide_lc)

arquitetura                acertos  buscas/caso  chamadas ao modelo/caso  tokens/caso  custo total (US$)  custo/acerto (US$)  segundos/caso
busca fixa                 16/16    1.0          1.0                      810          0.0194             0.00121             1.3          
agente decide              11/16    0.94         1.94                     3669         0.0753             0.00685             3.2          
busca fixa · LangChain     16/16    1.0          1.0                      1542         0.0337             0.00211             1.6          
agente decide · LangChain  16/16    0.94         1.94                     3502         0.0676             0.00423             3.6          

caso  categoria     busca fixa                        agente decide                     busca fixa · LangChain            agente decide · LangChain         
N1    normal        ok  b=1                           ok  b=1                           ok  b=1                           ok  b=1             

## 7 · O que o framework fez, e o que continuou sendo nosso

| | Notebook 03, sem framework | Notebook 04, LangChain |
|---|---|---|
| Loop do agente | ~20 linhas, todas à vista | `create_agent`: uma chamada; o loop roda dentro do framework |
| Critério de parada | `responder`, `escalar_para_rh` e `MAX_PASSOS`, explícitos | resposta estruturada e `recursion_limit` |
| Ver cada passo | `mostrar=True` no próprio loop | ler `saida["messages"]`, ou uma ferramenta de trace (Encontro 5) |
| Trocar de provedor | um arquivo, `comum/modelo.py` | uma classe por provedor (`ChatAnthropic`, `ChatOpenAI`) |
| Busca híbrida | 15 linhas de RRF nossas | `EnsembleRetriever` |
| Prefixos do E5 | nossos | nossos, num embrulho de `Embeddings` |
| Regra de vigência e autoridade | nossa | **nossa**: o framework não sabe o que é uma política revogada |
| Contrato da ferramenta | nosso | **o mesmo** |
| Validação, corte da saída, registro | `camada.py` | **a mesma** `camada.py` |

**Pergunta para a turma.**

- O placar mudou entre o 03 e o 04? Se mudou, a causa é o framework, o prompt de encerramento ou a variância entre rodadas?
- Que linha da tabela pesa mais para a equipe de vocês: escrever menos código, ou enxergar cada passo?
- Quando algo der errado em produção, em qual das duas versões vocês acham o passo que falhou mais rápido?

A escolha de framework, com critérios, é o tema do Encontro 4.